In [ ]:
# Imports
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os
import requests
import json

# Configuration
load_dotenv()
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# API Configuration
API_BASE_URL = os.getenv('API_BASE_URL', 'http://localhost:8000')

print(f"✅ Connecté à l'API : {API_BASE_URL}")

## 1. Création de Profils Santé de Test

In [ ]:
# Définir des profils types
test_profiles = {
    'diabetique': {
        'name': 'Patient Diabétique Type 2',
        'diseases': ['diabetes_type2'],
        'weight': 85,
        'activity_level': 'moderate',
        'concerns': ['index glycémique', 'sucres rapides']
    },
    'hypertendu': {
        'name': 'Patient Hypertendu',
        'diseases': ['hypertension'],
        'weight': 90,
        'activity_level': 'light',
        'concerns': ['sodium', 'sel']
    },
    'renal': {
        'name': 'Patient Insuffisance Rénale',
        'diseases': ['kidney_disease'],
        'weight': 70,
        'activity_level': 'light',
        'concerns': ['protéines', 'potassium', 'phosphore']
    },
    'sain': {
        'name': 'Profil Sain',
        'diseases': [],
        'weight': 75,
        'activity_level': 'moderate',
        'concerns': ['équilibre nutritionnel']
    }
}

pd.DataFrame(test_profiles).T

## 2. Récupération des Plats Disponibles

In [ ]:
# Récupérer la liste des plats
response = requests.get(f"{API_BASE_URL}/dishes", params={'limit': 20})
dishes = response.json()

df_dishes = pd.DataFrame(dishes)
print(f"📊 {len(df_dishes)} plats disponibles\n")
df_dishes[['name', 'meal_type', 'cuisine_origin']].head(10)

## 3. Analyse d'un Plat pour Différents Profils

In [ ]:
# Fonction pour analyser un plat
def analyze_dish_for_profile(dish_id, profile_name):
    """
    Analyse un plat pour un profil donné
    """
    # Note: En production, utiliser un vrai user_id du profil
    response = requests.post(
        f"{API_BASE_URL}/analyze-dish",
        json={
            'dish_id': dish_id,
            'user_id': f"test_{profile_name}"
        }
    )
    
    if response.status_code == 200:
        return response.json()
    else:
        return None

# Sélectionner un plat pour test
test_dish_id = df_dishes.iloc[0]['id']
test_dish_name = df_dishes.iloc[0]['name']

print(f"🍽️ Analyse du plat : {test_dish_name}")
print("=" * 60)

In [ ]:
# Simuler l'analyse pour différents profils
# Note: Ce code nécessite que les profils soient créés dans la DB

results = []

# Exemple de données simulées (remplacer par de vraies calls API)
simulated_results = {
    'diabetique': {'score': 45, 'alert_level': 'WARNING', 'problems': ['IG élevé', 'Sucres rapides']},
    'hypertendu': {'score': 55, 'alert_level': 'WARNING', 'problems': ['Sodium élevé']},
    'renal': {'score': 40, 'alert_level': 'DANGER', 'problems': ['Protéines élevées', 'Potassium élevé']},
    'sain': {'score': 85, 'alert_level': 'SAFE', 'problems': []}
}

for profile_name, data in simulated_results.items():
    results.append({
        'Profil': test_profiles[profile_name]['name'],
        'Score': data['score'],
        'Niveau Alerte': data['alert_level'],
        'Problèmes Identifiés': ', '.join(data['problems']) if data['problems'] else 'Aucun'
    })

df_results = pd.DataFrame(results)
df_results

## 4. Visualisation des Scores par Profil

In [ ]:
# Graphique des scores
fig, ax = plt.subplots(figsize=(12, 6))

colors = {'SAFE': 'green', 'WARNING': 'orange', 'DANGER': 'red'}
bar_colors = [colors[level] for level in df_results['Niveau Alerte']]

bars = ax.barh(df_results['Profil'], df_results['Score'], color=bar_colors, edgecolor='black')

# Zones de score
ax.axvline(70, color='green', linestyle='--', alpha=0.5, label='Seuil Safe (70)')
ax.axvline(50, color='orange', linestyle='--', alpha=0.5, label='Seuil Warning (50)')

ax.set_xlabel('Score de Santé', fontsize=12, fontweight='bold')
ax.set_title(f'Score de Compatibilité pour "{test_dish_name}" par Profil', fontsize=14, fontweight='bold')
ax.set_xlim(0, 100)
ax.legend()
ax.grid(axis='x', alpha=0.3)

# Ajouter valeurs
for bar in bars:
    width = bar.get_width()
    ax.text(width + 2, bar.get_y() + bar.get_height()/2, 
            f'{width:.0f}', ha='left', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Analyse des Règles Métier

In [ ]:
# Règles implémentées dans le système
business_rules = {
    'Diabète Type 2': [
        {'règle': 'Index Glycémique', 'seuil': 'IG < 55 → +points', 'poids': 'Élevé'},
        {'règle': 'Glucides simples', 'seuil': '< 10g/100g → +points', 'poids': 'Élevé'},
        {'règle': 'Fibres', 'seuil': '> 5g/100g → +points', 'poids': 'Moyen'},
    ],
    'Hypertension': [
        {'règle': 'Sodium', 'seuil': '< 300mg/100g → +points', 'poids': 'Très élevé'},
        {'règle': 'Potassium', 'seuil': '> 500mg/100g → +points', 'poids': 'Élevé'},
        {'règle': 'Ratio K/Na', 'seuil': '> 2 → +points', 'poids': 'Élevé'},
    ],
    'Insuffisance Rénale': [
        {'règle': 'Protéines', 'seuil': '< 15g/100g → +points', 'poids': 'Très élevé'},
        {'règle': 'Potassium', 'seuil': '< 400mg/100g → +points', 'poids': 'Très élevé'},
        {'règle': 'Phosphore', 'seuil': '< 200mg/100g → +points', 'poids': 'Élevé'},
    ]
}

# Affichage des règles
for disease, rules in business_rules.items():
    print(f"\n{'='*60}")
    print(f"📋 RÈGLES MÉTIER : {disease}")
    print('='*60)
    df_rules = pd.DataFrame(rules)
    print(df_rules.to_string(index=False))

## 6. Test du Système de Recommandation

In [ ]:
# Fonction pour obtenir des recommandations
def get_recommendations(user_id, meal_type=None, limit=5):
    """
    Obtient des recommandations personnalisées
    """
    params = {'limit': limit}
    if meal_type:
        params['meal_type'] = meal_type
    
    response = requests.get(
        f"{API_BASE_URL}/recommendations/{user_id}",
        params=params
    )
    
    if response.status_code == 200:
        return response.json()
    else:
        return None

# Exemple de recommandations (simulé)
simulated_recommendations = [
    {'dish_name': 'Salade de quinoa', 'score': 92, 'reason': 'IG bas, riche en fibres'},
    {'dish_name': 'Mafé aux légumes', 'score': 85, 'reason': 'Équilibré en macronutriments'},
    {'dish_name': 'Poisson grillé', 'score': 88, 'reason': 'Faible en glucides, riche en protéines'},
    {'dish_name': 'Thiéboudienne légère', 'score': 75, 'reason': 'Version adaptée, moins de sel'},
    {'dish_name': 'Soupe de légumes', 'score': 90, 'reason': 'Faible densité calorique, nutritive'}
]

df_reco = pd.DataFrame(simulated_recommendations)
print("\n🎯 TOP 5 RECOMMANDATIONS (Profil Diabétique)\n")
df_reco

In [ ]:
# Visualisation des recommandations
plt.figure(figsize=(12, 6))
bars = plt.barh(df_reco['dish_name'], df_reco['score'], color='steelblue', edgecolor='black')

# Gradient de couleur selon score
colors_gradient = plt.cm.RdYlGn(df_reco['score'] / 100)
for bar, color in zip(bars, colors_gradient):
    bar.set_color(color)

plt.xlabel('Score de Recommandation', fontsize=12, fontweight='bold')
plt.title('Top Recommandations Personnalisées', fontsize=14, fontweight='bold')
plt.xlim(0, 100)
plt.grid(axis='x', alpha=0.3)

# Ajouter valeurs
for i, (score, reason) in enumerate(zip(df_reco['score'], df_reco['reason'])):
    plt.text(score + 1, i, f"{score}", va='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 7. Évaluation de la Qualité des Recommandations

In [ ]:
# Métriques de qualité
print("\n" + "="*60)
print("📊 MÉTRIQUES DE QUALITÉ DU SYSTÈME DE RECOMMANDATION")
print("="*60)

metrics = {
    'Score Moyen': df_reco['score'].mean(),
    'Score Médian': df_reco['score'].median(),
    'Score Min': df_reco['score'].min(),
    'Score Max': df_reco['score'].max(),
    'Écart-type': df_reco['score'].std(),
    '% Recommandations Safe (>70)': (df_reco['score'] > 70).sum() / len(df_reco) * 100
}

for metric, value in metrics.items():
    print(f"{metric:.<40} {value:.2f}")

print("\n✅ Toutes les recommandations sont au-dessus du seuil de sécurité")

## 8. Conclusions & Améliorations Futures

### Points Forts :

1. ✅ **Personnalisation** : Scores adaptés au profil santé
2. ✅ **Transparence** : Explications claires des recommandations
3. ✅ **Sécurité** : Système d'alertes multi-niveaux
4. ✅ **Exhaustivité** : Prise en compte de multiples pathologies

### Améliorations Possibles :

1. 🔄 **Machine Learning** : Intégrer un modèle d'apprentissage supervisé
2. 🔄 **Historique** : Prendre en compte les préférences utilisateur
3. 🔄 **Filtrage Collaboratif** : Recommandations basées sur des profils similaires
4. 🔄 **Optimisation Multi-critères** : Équilibrer goût, santé, et disponibilité

### Métriques d'Évaluation :

- **Précision** : Score de santé > 70 pour 90% des recommandations
- **Diversité** : Variété des types de plats recommandés
- **Satisfaction** : Feedback utilisateur (à implémenter)
- **Performance** : Temps de réponse < 500ms